[Open in Colab](https://colab.research.google.com/github/fcoliveira-utfpr/aquacrop_ml/blob/main/11_assistente_perguntas.ipynb)

# Consulta rápida — aquacrop_ml

Formulário para pesquisadores consultarem os resultados do projeto (melhor data de semeadura,
produtividade prevista, custos de produção, preços) sem precisar mexer nos outros notebooks.
Chama **as mesmas funções** que viraram tools do servidor MCP (`mcp_maiz/data.py` e
`mcp_maiz/pipeline.py`) — aqui, em vez de perguntar em texto livre, você escolhe o tipo de consulta
num menu e preenche alguns campos. Não precisa de chave de API nem gera custo nenhum.

**Para usar**: `Arquivo > Salvar uma cópia no Drive` (assim você edita sua própria cópia) e rode as
células em ordem.


## 1. Setup

Clona o repositório (se estiver rodando no Colab), instala o SDK da Anthropic e disponibiliza
`mcp_maiz/data.py`/`pipeline.py` para importar.

In [1]:
import os
import sys

RODANDO_NO_COLAB = "google.colab" in str(get_ipython())

if RODANDO_NO_COLAB:
    if not os.path.exists("aquacrop_ml"):
        !git clone -q https://github.com/fcoliveira-utfpr/aquacrop_ml.git
    if os.path.basename(os.getcwd()) != "aquacrop_ml":
        os.chdir("aquacrop_ml")

sys.path.insert(0, "mcp_maiz")
print("Setup OK — rodando no Colab" if RODANDO_NO_COLAB else "Setup OK — rodando localmente")

Setup OK — rodando localmente


## 2. Funções de consulta (mesmas do servidor MCP)

`data.py` responde instantaneamente (lê os CSVs já prontos). `pipeline.py` roda a simulação ao
vivo (clima + balanço hídrico + modelo) para qualquer município do Brasil — mais lenta, pode levar
dezenas de segundos.

In [2]:
import data
import pipeline

print("Módulos carregados.")

Módulos carregados.


## 3. Interface

Escolha o tipo de consulta no menu — os campos abaixo mudam de acordo. Campos numéricos com valor
`0` são tratados como "sem filtro" (traz tudo).

In [3]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

municipios_oeste = data.listar_municipios_oeste()
cidades_custo = data.listar_categorias_custo()["cidades"]

# --- campos (reaproveitados entre consultas que pedem a mesma coisa) ---
w_municipio_oeste = widgets.Dropdown(options=municipios_oeste, description="Município:")
w_metodo = widgets.Dropdown(options=["integrado", "topsis"], value="topsis", description="Método:")
w_ano_prod = widgets.IntText(value=0, description="Ano (0=todos):")
w_municipio_livre = widgets.Text(placeholder="qualquer cidade do Brasil", description="Município:")
w_ano_livre = widgets.IntText(value=2023, description="Ano:")
w_data_sem = widgets.Dropdown(
    options=["05/02", "15/02", "25/02", "05/03", "15/03", "25/03"], value="15/03", description="Semeadura:"
)
w_cidade_custo = widgets.Dropdown(options=cidades_custo, description="Cidade:")
w_ano_ini = widgets.IntText(value=0, description="Ano início (0=sem filtro):", style={"description_width": "160px"})
w_ano_fim = widgets.IntText(value=0, description="Ano fim (0=sem filtro):", style={"description_width": "160px"})
w_categoria = widgets.Text(placeholder="ex: Fertilizantes (opcional)", description="Categoria:")
w_apenas_total = widgets.Checkbox(value=True, description="Só o Custo Total (J)")
w_deflacionado = widgets.Checkbox(value=True, description="Valores reais (R$ de 2025)")
w_safra = widgets.Text(placeholder="ex: 2023/24 (opcional)", description="Safra:")
w_fonte = widgets.Dropdown(options=["deral", "ipea"], description="Fonte:")
w_ano_ipca = widgets.IntText(value=2015, description="Ano:")
w_ano_base = widgets.IntText(value=2025, description="Ano-base:")

GRUPOS = {
    "Melhor data de semeadura (Oeste do PR)": widgets.VBox([w_municipio_oeste, w_metodo]),
    "Comparar as 6 datas de semeadura (Oeste do PR)": widgets.VBox([w_municipio_oeste, w_metodo]),
    "Produtividade já prevista (Oeste do PR)": widgets.VBox([w_municipio_oeste, w_ano_prod]),
    "Prever produtividade — qualquer município do Brasil (mais lenta)": widgets.VBox(
        [w_municipio_livre, w_ano_livre, w_data_sem]
    ),
    "Custo de produção (CONAB)": widgets.VBox(
        [w_cidade_custo, w_ano_ini, w_ano_fim, w_categoria, w_apenas_total, w_deflacionado]
    ),
    "Custo de referência SEAB/DERAL": widgets.VBox([w_safra]),
    "Preço do milho": widgets.VBox([w_ano_ini, w_ano_fim, w_fonte]),
    "Fator de deflação IPCA": widgets.VBox([w_ano_ipca, w_ano_base]),
}

w_tipo = widgets.Dropdown(options=list(GRUPOS.keys()), description="Consulta:", layout=widgets.Layout(width="480px"))
painel_campos = widgets.Output()
saida = widgets.Output()
botao = widgets.Button(description="Consultar", button_style="primary")


def mostrar_campos(*_):
    painel_campos.clear_output()
    with painel_campos:
        display(GRUPOS[w_tipo.value])


def exibir_resultado(_):
    saida.clear_output()
    tipo = w_tipo.value
    with saida:
        try:
            if tipo == "Melhor data de semeadura (Oeste do PR)":
                resultado = data.melhor_data_semeadura(w_municipio_oeste.value, w_metodo.value)
            elif tipo == "Comparar as 6 datas de semeadura (Oeste do PR)":
                resultado = data.matriz_risco_municipio(w_municipio_oeste.value, w_metodo.value)
            elif tipo == "Produtividade já prevista (Oeste do PR)":
                resultado = data.produtividade_prevista(w_municipio_oeste.value, w_ano_prod.value or None)
            elif tipo == "Prever produtividade — qualquer município do Brasil (mais lenta)":
                print("Rodando simulação ao vivo (clima + modelo) — pode levar de 10 a 60 segundos...")
                resultado = pipeline.prever_produtividade(w_municipio_livre.value, w_ano_livre.value, w_data_sem.value)
            elif tipo == "Custo de produção (CONAB)":
                resultado = data.custo_producao(
                    w_cidade_custo.value,
                    ano_inicio=w_ano_ini.value or None,
                    ano_fim=w_ano_fim.value or None,
                    categoria=w_categoria.value or None,
                    apenas_total=w_apenas_total.value,
                    deflacionado=w_deflacionado.value,
                )
            elif tipo == "Custo de referência SEAB/DERAL":
                resultado = data.custo_deral_safrinha(w_safra.value or None)
            elif tipo == "Preço do milho":
                resultado = data.preco_milho(w_ano_ini.value or None, w_ano_fim.value or None, w_fonte.value)
            elif tipo == "Fator de deflação IPCA":
                resultado = data.fator_deflator_ipca(w_ano_ipca.value, w_ano_base.value)

            if isinstance(resultado, list):
                display(pd.DataFrame(resultado))
            elif isinstance(resultado, dict):
                display(pd.DataFrame([resultado]))
            else:
                print(resultado)
        except Exception as exc:
            print(f"Erro: {exc}")


w_tipo.observe(mostrar_campos, names="value")
botao.on_click(exibir_resultado)

mostrar_campos()
display(w_tipo, painel_campos, botao, saida)

Dropdown(description='Consulta:', layout=Layout(width='480px'), options=('Melhor data de semeadura (Oeste do P…

Output()

Button(button_style='primary', description='Consultar', style=ButtonStyle())

Output()